In [91]:
import pandas as pd
import numpy as np


In [92]:
df = pd.read_csv('../../data/stock_prices.csv')

Data Exploration

In [93]:
df.describe()

,open,high,low,close,volume
count,71044.000000,71044.000000,71044.000000,71044.000000,7.104400e+04
mean,195.730367,199.351455,192.178036,195.974734,6.093784e+05
std,710.294334,721.000404,699.491964,709.825037,3.720481e+06
min,0.100000,0.100000,0.100000,0.100000,0.000000e+00
25%,15.700000,16.000000,15.300000,15.700000,3.983000e+03
50%,44.300000,45.000000,43.500000,44.400000,4.410350e+04
75%,114.500000,116.367958,112.093584,114.500000,2.661955e+05
max,18750.000000,18750.000000,18750.000000,18750.000000,3.554465e+08


In [94]:
df.head()

,symbol,date,open,high,low,close,volume
0,ABAN.N0000,2025-02-03,468.0,485.0,447.50,470.00,10154.0
1,ABAN.N0000,2025-02-05,460.0,460.0,430.00,433.00,14587.0
2,ABAN.N0000,2025-02-06,437.0,475.0,430.00,440.75,5961.0
3,ABAN.N0000,2025-02-07,455.0,470.0,451.00,465.00,2629.0
4,ABAN.N0000,2025-02-10,455.0,468.0,441.75,468.00,1763.0


In [95]:
df.tail()

,symbol,date,open,high,low,close,volume
71039,VPEL.N0000,2026-02-23,16.4,16.5,16.0,16.0,476285.0
71040,VPEL.N0000,2026-02-24,16.0,16.0,15.4,15.8,102412.0
71041,VPEL.N0000,2026-02-25,15.9,15.9,15.5,15.5,155118.0
71042,VPEL.N0000,2026-02-26,15.7,15.7,15.4,15.5,55255.0
71043,VPEL.N0000,2026-02-27,15.4,15.5,15.0,15.2,262528.0


Splitting symbol and suffix

In [96]:
split = df['symbol'].str.split('.')
df['base_symbol'] = split.str[0]   # e.g. 'CIC'  — for display/joining only
df['suffix']      = split.str[1]   # e.g. 'N0000'

In [97]:
df

,symbol,date,open,high,low,close,volume,base_symbol,suffix
0,ABAN.N0000,2025-02-03,468.0,485.0,447.50,470.00,10154.0,ABAN,N0000
1,ABAN.N0000,2025-02-05,460.0,460.0,430.00,433.00,14587.0,ABAN,N0000
2,ABAN.N0000,2025-02-06,437.0,475.0,430.00,440.75,5961.0,ABAN,N0000
3,ABAN.N0000,2025-02-07,455.0,470.0,451.00,465.00,2629.0,ABAN,N0000
4,ABAN.N0000,2025-02-10,455.0,468.0,441.75,468.00,1763.0,ABAN,N0000
...,...,...,...,...,...,...,...,...,...
71039,VPEL.N0000,2026-02-23,16.4,16.5,16.00,16.00,476285.0,VPEL,N0000
71040,VPEL.N0000,2026-02-24,16.0,16.0,15.40,15.80,102412.0,VPEL,N0000
71041,VPEL.N0000,2026-02-25,15.9,15.9,15.50,15.50,155118.0,VPEL,N0000
71042,VPEL.N0000,2026-02-26,15.7,15.7,15.40,15.50,55255.0,VPEL,N0000


In [98]:
df.isnull().sum()

symbol         0
date           0
open           0
high           0
low            0
close          0
volume         0
base_symbol    0
suffix         0
dtype: int64

In [99]:
# Load sector mapping — keyed on base symbol (e.g. 'COMB')
# We extract the base symbol temporarily for the lookup only.
# The full symbol (e.g. 'COMB.N0000') is kept as the unique identifier.

sector_map_df = pd.read_csv('../../data/sector_mapping.csv')
sector_map    = sector_map_df.set_index('symbol')['sector']

base_symbol   = df['symbol'].str.split('.').str[0]   # 'COMB.N0000' → 'COMB'
df['sector']  = base_symbol.map(sector_map).fillna('Others')

unmapped = df[df['sector'] == 'Others']['symbol'].unique()
base_names = sorted(set(s.split('.')[0] for s in unmapped))
print(f'{len(unmapped)} symbols not in mapping: {base_names}')
print()
print('Sector distribution (unique stocks):')
print(df.groupby('sector')['symbol'].nunique().sort_values(ascending=False).to_string())

4 symbols not in mapping: ['BBH', 'CALC', 'CALI', 'CSF']

Sector distribution (unique stocks):
sector
Food & Beverage               45
Diversified Financials        42
Consumer Services             33
Capital Goods                 30
Materials                     24
Real Estate                   17
Banks                         17
Insurance                     14
Retailing                     14
Utilities                      9
Healthcare Equipment           9
Consumer Durables              8
Commercial Services            7
Food Retailing                 4
Others                         4
Transportation                 3
Energy                         3
Telecommunication Services     2
Household Products             2
Automobiles                    1
Software & Services            1


In [100]:
df.to_csv('../../output/processed_data/01_sector_handled.csv',index = False)